# Phase 4: Feature Management

This notebook creates a custom feature store for the recommendation system.

It generates reusable user, item, and interaction features using the training dataset.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"

PROCESSED_DATA_DIR = os.path.join(
    PROJECT_DIR, "data", "processed"
)

FEATURES_DIR = os.path.join(
    PROJECT_DIR, "data", "features"
)

os.makedirs(FEATURES_DIR, exist_ok=True)

train_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "train_data.parquet")
)

print("Training data loaded:", train_df.shape)
display(train_df.head())

Mounted at /content/drive
Training data loaded: (80668, 13)


,userId,movieId,rating,timestamp,datetime,year,month,day,title,genres,age,occupation,location
0,429,165,4.0,828124615,1996-03-29 18:36:55,1996,3,29,Die Hard: With a Vengeance (1995),Action|Crime|Thriller,34,Doctor,Vancouver
1,429,595,5.0,828124615,1996-03-29 18:36:55,1996,3,29,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,34,Doctor,Vancouver
2,429,434,4.0,828124615,1996-03-29 18:36:55,1996,3,29,Cliffhanger (1993),Action|Adventure|Thriller,34,Doctor,Vancouver
3,429,590,5.0,828124615,1996-03-29 18:36:55,1996,3,29,Dances with Wolves (1990),Adventure|Drama|Western,34,Doctor,Vancouver
4,429,588,5.0,828124615,1996-03-29 18:36:55,1996,3,29,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,34,Doctor,Vancouver


In [2]:
# Create User features
user_features = train_df.groupby("userId").agg(
    user_avg_rating=("rating", "mean"),
    user_activity_count=("rating", "count")
).reset_index()

print("User features created:", user_features.shape)
display(user_features.head())

User features created: (522, 3)


,userId,user_avg_rating,user_activity_count
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44


In [3]:
#Create movie/item features
item_features = train_df.groupby("movieId").agg(
    item_avg_rating=("rating", "mean"),
    item_popularity=("rating", "count"),
    unique_user_count=("userId", "nunique")
).reset_index()

print("Item features created:", item_features.shape)
display(item_features.head())

# Create interaction features
interaction_features = train_df[
    [
        "userId",
        "movieId",
        "rating",
        "timestamp",
        "year",
        "month",
        "day"
    ]
].copy()

# Save features in CSV and Parquet formats

feature_datasets = {
    "user_features": user_features,
    "item_features": item_features,
    "interaction_features": interaction_features
}

for name, dataframe in feature_datasets.items():

    dataframe.to_csv(
        os.path.join(FEATURES_DIR, name + ".csv"),
        index=False
    )

    dataframe.to_parquet(
        os.path.join(FEATURES_DIR, name + ".parquet"),
        index=False
    )

    print(name, "saved successfully:", dataframe.shape)


Item features created: (7867, 4)


,movieId,item_avg_rating,item_popularity,unique_user_count
0,1,3.923913,184,184
1,2,3.400000,95,95
2,3,3.294118,51,51
3,4,2.357143,7,7
4,5,3.063830,47,47


user_features saved successfully: (522, 3)
item_features saved successfully: (7867, 4)
interaction_features saved successfully: (80668, 7)
